### Evoluzione della qualità: Come sono cambiati i voti medi (Ratings.csv) degli anime nel corso degli anni (collegando ai dati temporali degli anime)? C'è stato un "periodo d'oro"?

In [116]:
import duckdb
import pandas as pd
import plotly.express as px
from lib import readSaveCsv as rsc
from lib import dbconnection

engine = dbconnection.create_db_engine()

con = duckdb.connect(database=':memory:')
details = rsc.import_data(engine, "details_cleaned")

Successo: Dati letti correttamente.


In [117]:
PATH_RATINGS = "../../../cleaned_data/ratings_cleaned.parquet"

query = f"""
SELECT
    d.year,
    AVG(r.score) AS avg_score,
    COUNT(r.score) AS total_valid_ratings
FROM '{PATH_RATINGS}' AS r
JOIN details AS d ON r.anime_id = d.mal_id
WHERE d.year IS NOT NULL
  AND r.score > 0
GROUP BY d.year
HAVING COUNT(r.score) > 50
ORDER BY d.year
"""

In [118]:
df_golden_age = con.execute(query).df()
df_golden_age.head()

,year,avg_score,total_valid_ratings
0,1961.0,6.269231,52
1,1963.0,6.462179,1441
2,1964.0,6.015686,255
3,1965.0,6.186104,1209
4,1966.0,6.313636,880


In [119]:
fig = px.line(df_golden_age, x='year', y='avg_score',
              title='Average score each year',
              labels={'avg_score': 'Average score', 'year': 'Year'},
              markers=True)

In [120]:
fig.update_yaxes(rangemode="tozero")
fig.update_traces(mode='lines+markers')

fig.write_html("../../graphs/avgScoreTime2.html")
fig.show()

In [121]:
fig.update_yaxes(rangemode="normal")
fig.write_html("../../graphs/avgScoreTime1.html")
fig.show()

In [122]:
query_1964 = f"""
SELECT
    d.title,
    d.type,
    d.episodes,
    d.status,
    d.genres,
    d.studios,
    d.score AS score_originale,
    COUNT(r.score) AS voti_effettivi_nel_db,
    AVG(r.score) AS media_voti_calcolata
FROM details AS d
LEFT JOIN '{PATH_RATINGS}' AS r ON d.mal_id = r.anime_id
WHERE d.year = 1964
  AND (r.score > 0 OR r.score IS NULL)
GROUP BY 1, 2, 3, 4, 5, 6, 7
ORDER BY media_voti_calcolata DESC;
"""

In [123]:
df_1964 = con.execute(query_1964).df()
df_1964

,title,type,episodes,status,genres,studios,score_originale,voti_effettivi_nel_db,media_voti_calcolata
0,Shisukon Ouji,TV,14.0,Finished Airing,['Adventure'],[],NaN,23,7.217391
1,0-sen Hayato,TV,38.0,Finished Airing,[],[],5.96,66,6.227273
2,Shounen Ninja Kaze no Fujimaru,TV,65.0,Finished Airing,"['Action', 'Adventure']",['Toei Animation'],5.88,75,5.920000
3,Big X,TV,59.0,Finished Airing,"['Action', 'Sci-Fi']",['Tokyo Movie Shinsha'],6.07,91,5.637363


We can see a general growth in the average score per year. But is that because animes actually got better or are there any other factors? Let's see the same graph knowing the number of animes created each year.

In [124]:
query_qty = f"""
SELECT
    d.year,
    AVG(r.score) AS avg_score,
    COUNT(DISTINCT d.mal_id) AS anime_count
FROM '{PATH_RATINGS}' AS r
JOIN details AS d ON r.anime_id = d.mal_id
WHERE d.year IS NOT NULL
  AND r.score > 0
GROUP BY d.year
HAVING COUNT(r.score) > 50
ORDER BY d.year
"""

In [125]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

df_dual = con.execute(query_qty).df()
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Scatter(x=df_dual['year'], y=df_dual['avg_score'],
               name="Average score", mode='lines+markers'),
    secondary_y=False,
)
fig.add_trace(
    go.Bar(x=df_dual['year'], y=df_dual['anime_count'],
           name="Num. of created animes", opacity=0.3),
    secondary_y=True,
)

In [126]:
fig.update_yaxes(rangemode="tozero", secondary_y=False)

Analisi dei migliori anime per ciascun anno

In [127]:
query_top_performers = f"""
WITH anime_yearly_scores AS (
    SELECT
        d.year,
        d.mal_id,
        AVG(r.score) as individual_avg,
        COUNT(r.score) as vote_count
    FROM details d
    JOIN '{PATH_RATINGS}' r ON d.mal_id = r.anime_id
    WHERE r.score > 0 AND d.year IS NOT NULL
    GROUP BY d.year, d.mal_id
    HAVING COUNT(r.score) > 10
),
ranked_anime AS (
    SELECT
        year,
        individual_avg,
        PERCENT_RANK() OVER (PARTITION BY year ORDER BY individual_avg DESC) as percentile
    FROM anime_yearly_scores
)
SELECT
    year,
    AVG(individual_avg) as top_10_percent_avg
FROM ranked_anime
WHERE percentile <= 0.1
GROUP BY year
ORDER BY year
"""

In [128]:
df_top_10 = con.execute(query_top_performers).df()

In [129]:
fig = px.line(df_top_10, x='year', y='top_10_percent_avg',
              title='Qualità dei "Top Performer" (Media del miglior 10% per anno)',
              labels={'top_10_percent_avg': 'Media Top 10%', 'year': 'Anno'},
              markers=True)

In [130]:
fig.update_yaxes(rangemode="tozero")
fig.write_html('../../graphs/topPerformer.html')
fig.show()

In [131]:
import numpy as np

df_fit = df_top_10.copy()
df_fit['year'] = pd.to_numeric(df_fit['year'], errors='coerce')
df_fit = df_fit[df_fit['year'] < 2026].copy()

x = df_fit['year']
y = df_fit['top_10_percent_avg']

slope, intercept = np.polyfit(x, y, 1)
trendline_y = slope * x + intercept

In [132]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_fit['year'], y=df_fit['top_10_percent_avg'],
    mode='lines+markers', name='Media Top 10%'
))

fig.add_trace(go.Scatter(
    x=x, y=trendline_y,
    mode='lines', name='Trend Lineare',
    line=dict(color='red', dash='dot'),
    customdata=np.full(len(x), slope),
    hovertemplate="<b>Trend Lineare</b><br>Pendenza: %{customdata:.4f} punti/anno<extra></extra>"
))

fig.update_layout(
    title='Qualità dei Top Performer con Trend Corretto',
    xaxis_title='Anno', yaxis_title='Voto Medio Top 10%',
    yaxis=dict(rangemode="tozero")
)

fig.update_yaxes(rangemode="tozero")
fig.show()

fig.write_html("../../graphs/topPerformer_trend.html")